# Preprocessing

This notebook serves as a preprocessng step for the new raw dataset. 

Python was used after an earlier noteboo in R was unfathomably slow. 

Note that adding ARIMA style features is not implemented. This will be done on the notebooks for individual models if required. 

Tasks:
* Load the dataset (done)
* Column conversion (dates, numerics) (done)
* Assigns the id column: country_place with spaces separated by "_" (done)
* Assigning the conflict_indicator target variable. (done)
* Merging in additional data (elections, country facts) (done)
* Deletion of places with 0 local facts throughout all instances: these are likely to be admin3-admin1 records with events actually recorded at a place WITHIN that place. (done)
* Deletion of the final period (which is not likely to be a full 28 day period) (done)
* Splitting into train/test splits; testing data is the last 12 periods. (done)
* Saving the dataset as a new csv file. 
    

## Load the dataset:

In [1]:
import pandas as pd

df = pd.read_csv("Dataset Original/eastafrica3_2026-09-20_Raw.csv")


In [2]:
print("Rows x columns:", df.shape)

Rows x columns: (2332960, 36)


In [3]:
df.head()

,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,LocalDistinctActorCount,...,RegionalTotalFatalities,RegionalAvgSeverity,RegionalProtestersEventCount,RegionalStateForcesEventCount,RegionalPoliticalMilitiaEventCount,RegionalIdentityMilitiaEventCount,RegionalCiviliansEventCount,RegionalRebelGroupEventCount,RegionalRiotersEventCount,RegionalOtherTypeEventCount
0,South Sudan,South Sudan,6.7984,33.1308,103.426655,276.213349,2015-01-01T00:00:00,2015-01-29T00:00:00,0,1,...,0,0.0,0,0,0,0,0,0,0,0
1,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01T00:00:00,2015-01-29T00:00:00,0,1,...,0,0.0,0,0,0,0,0,0,0,0
2,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01T00:00:00,2015-01-29T00:00:00,0,1,...,0,0.0,0,0,0,0,0,0,0,0
3,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01T00:00:00,2015-01-29T00:00:00,0,1,...,0,0.0,0,0,0,0,0,0,0,0
4,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01T00:00:00,2015-01-29T00:00:00,0,1,...,0,0.0,0,0,0,0,0,0,0,0


In [4]:
unique_countries = df["country"].unique()
print(unique_countries)
print("Countries: ", len(unique_countries))

unique_years = sorted(df['periodStart'].str.slice(0, 4).unique())
print(unique_years)
print("Years: ", unique_years)

['South Sudan' 'Uganda' 'Sudan' 'Ethiopia' 'Somalia' 'Kenya' 'Libya'
 'Central African Republic' 'Egypt' 'Chad']
Countries:  10
['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Years:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


## Date Conversion

In [5]:
df['periodStart'] = pd.to_datetime(df['periodStart'], format ='%Y-%m-%dT%H:%M:%S')
df['periodEnd'] = pd.to_datetime(df['periodEnd'], format ='%Y-%m-%dT%H:%M:%S')

In [6]:
df.head(2)

,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,LocalDistinctActorCount,...,RegionalTotalFatalities,RegionalAvgSeverity,RegionalProtestersEventCount,RegionalStateForcesEventCount,RegionalPoliticalMilitiaEventCount,RegionalIdentityMilitiaEventCount,RegionalCiviliansEventCount,RegionalRebelGroupEventCount,RegionalRiotersEventCount,RegionalOtherTypeEventCount
0,South Sudan,South Sudan,6.7984,33.1308,103.426655,276.213349,2015-01-01,2015-01-29,0,1,...,0,0.0,0,0,0,0,0,0,0,0
1,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01,2015-01-29,0,1,...,0,0.0,0,0,0,0,0,0,0,0


### Identify the min and max periods:



In [7]:
min_start = print("Overall periodStart range: ", df['periodStart'].min(), " - ", df['periodStart'].max())
print("Periods:", len(df["periodStart"].unique()))

Overall periodStart range:  2015-01-01 00:00:00  -  2025-08-28 00:00:00
Periods: 140


## Numeric conversion

In [8]:
## Numeric Conversaion, also sets any missing values to NaN
df['latitude'] = pd.to_numeric(df['latitude'], errors = 'coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors = 'coerce')

## ID column

In [9]:
df['id'] = (df['country'].str.replace(" ", "_")
 + "_" + df['name']).str.replace(" ", "_")

cols = ['id'] + [c for c in df.columns if c != 'id']
df = df[cols]

df.head()

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,RegionalTotalFatalities,RegionalAvgSeverity,RegionalProtestersEventCount,RegionalStateForcesEventCount,RegionalPoliticalMilitiaEventCount,RegionalIdentityMilitiaEventCount,RegionalCiviliansEventCount,RegionalRebelGroupEventCount,RegionalRiotersEventCount,RegionalOtherTypeEventCount
0,South_Sudan_South_Sudan,South Sudan,South Sudan,6.7984,33.1308,103.426655,276.213349,2015-01-01,2015-01-29,0,...,0,0.0,0,0,0,0,0,0,0,0
1,South_Sudan_Lakes,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01,2015-01-29,0,...,0,0.0,0,0,0,0,0,0,0,0
2,South_Sudan_Rumbek_East,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01,2015-01-29,0,...,0,0.0,0,0,0,0,0,0,0,0
3,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01,2015-01-29,0,...,0,0.0,0,0,0,0,0,0,0,0
4,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01,2015-01-29,0,...,0,0.0,0,0,0,0,0,0,0,0


### Unique Ids

In [10]:
unique_ids = df["id"].unique()
print("Unique Ids: ", len(unique_ids))

Unique Ids:  16652


## Assigning conflict_indicator

In [11]:
import numpy as np

local_cond = (df['LocalAvgSeverity'] >= 3) | (df['LocalTotalFatalities'] >= 3)
regional_cond = (df['RegionalAvgSeverity'] >= 3) | (df['RegionalTotalFatalities'] >= 3)

df['conflict_indicator'] = np.select(
    [
        local_cond & regional_cond,   # both
        local_cond,                   # local only
        regional_cond                 # regional only
    ],
    [
        3,  # both conditions met
        1,  # local only
        2   # regional only
    ],
    default=0
)

In [12]:
df.head(5)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,RegionalAvgSeverity,RegionalProtestersEventCount,RegionalStateForcesEventCount,RegionalPoliticalMilitiaEventCount,RegionalIdentityMilitiaEventCount,RegionalCiviliansEventCount,RegionalRebelGroupEventCount,RegionalRiotersEventCount,RegionalOtherTypeEventCount,conflict_indicator
0,South_Sudan_South_Sudan,South Sudan,South Sudan,6.7984,33.1308,103.426655,276.213349,2015-01-01,2015-01-29,0,...,0.0,0,0,0,0,0,0,0,0,0
1,South_Sudan_Lakes,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01,2015-01-29,0,...,0.0,0,0,0,0,0,0,0,0,0
2,South_Sudan_Rumbek_East,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01,2015-01-29,0,...,0.0,0,0,0,0,0,0,0,0,0
3,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01,2015-01-29,0,...,0.0,0,0,0,0,0,0,0,0,0
4,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01,2015-01-29,0,...,0.0,0,0,0,0,0,0,0,0,0


### Overall Class Breakdown

In [13]:
breakdown = (
    df['conflict_indicator']
      .value_counts()
      .sort_index()
      .rename_axis('conflict_indicator')
      .reset_index(name='count')
)

# Add percentage column
total = len(df)
breakdown['percentage'] = (breakdown['count'] / total * 100).round(2)

print(breakdown)


   conflict_indicator    count  percentage
0                   0  2295450       98.39
1                   1    35136        1.51
2                   2       21        0.00
3                   3     2353        0.10


### Id Breakdown by Maximum conflict_indicator

In [14]:
id_max = (
    df.groupby('id')['conflict_indicator']
      .max()
      .reset_index()
)

breakdown = (
    id_max['conflict_indicator']
        .value_counts()
        .sort_index()
        .rename_axis('conflict_indicator')
        .reset_index(name='count')
)

total_ids = id_max.shape[0]
breakdown['percentage'] = (breakdown['count'] / total_ids * 100).round(2)

print(breakdown)

   conflict_indicator  count  percentage
0                   0   7868       47.25
1                   1   8531       51.23
2                   2      1        0.01
3                   3    252        1.51


In [15]:

print("missing values in minCapitalDistanceKm before processing: ", df['minCapitalDistanceKm'].replace("", pd.NA).isna().sum())
df['minCapitalDistanceKm'] = pd.to_numeric(df['minCapitalDistanceKm'])
df['minCapitalDistanceKm'].replace("", pd.NA).isna().sum()
print("missing values in minCapitalDistanceKm after processing: ", df['minCapitalDistanceKm'].replace("", pd.NA).isna().sum())

print("missing values in minBorderDistanceKm before processing: ", df['minBorderDistanceKm'].replace("", pd.NA).isna().sum())
df['minBorderDistanceKm'] = pd.to_numeric(df['minBorderDistanceKm'])
df['minBorderDistanceKm'].replace("", pd.NA).isna().sum()
print("missing values in minBorderDistanceKm after processing: ", df['minBorderDistanceKm'].replace("", pd.NA).isna().sum())


missing values in minCapitalDistanceKm before processing:  26320
missing values in minCapitalDistanceKm after processing:  26320
missing values in minBorderDistanceKm before processing:  26320
missing values in minBorderDistanceKm after processing:  26320


## Merging preprocessed country facts data

In [16]:
df_country_facts_cleaned = pd.read_csv("Country Data Preprocessed/Country Facts.csv")

In [17]:
df_country_facts_cleaned.head(4)

,Country Name,Country Code,Year,Time Code,"Population, total [SP.POP.TOTL]",Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST],Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST],"Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]",Children out of school (% of primary school age) [SE.PRM.UNER.ZS],GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD],...,GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag,"Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag",Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag,Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag,"Population, total [SP.POP.TOTL]_missing_flag","Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag",Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag,Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag"
0,Central African Republic,NaN,1990,YR1990,2871910,-1.260176,-1.226337,32.014149,41.592239,387.280283,...,0,1,1,1,0,0,0,0,1,0
1,Central African Republic,NaN,2000,YR2000,3833416,-1.260176,-1.226337,32.014149,41.592239,387.280283,...,0,1,1,1,0,0,1,0,0,0
2,Central African Republic,NaN,2016,YR2016,4713663,-1.308185,-1.688903,38.480461,41.592239,387.280283,...,0,0,0,0,0,0,0,0,0,0
3,Central African Republic,NaN,2017,YR2017,4793511,-1.180782,-1.605281,38.480461,41.592239,408.750638,...,0,0,0,0,0,0,1,0,0,0


In [18]:
list(df_country_facts_cleaned.columns)

cols_to_use = ['Country Name', 'Year',
    'Population, total [SP.POP.TOTL]',
 'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]',
 'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]',
 'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]',
 'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]',
 'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]',
 'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]',
 'Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]',
 'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]',
 'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]',
 'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]',
 'Permanent cropland (% of land area) [AG.LND.CROP.ZS]',
 'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]',
 'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]',
 'Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]_missing_flag',
 'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]_missing_flag',
 'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]_missing_flag',
 'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]_missing_flag',
 'Country Code_missing_flag',
 'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag',
 'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag',
 'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag',
 'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag',
 'Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag',
 'Population, total [SP.POP.TOTL]_missing_flag',
 'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag',
 'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag',
 'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag',
 'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag']

In [19]:
import itertools

#add a year column to df
df["Year"] = df['periodStart'].dt.year

df = df.merge(
    df_country_facts_cleaned[cols_to_use],
    how='left',
    left_on=['country', 'Year'],      #df
    right_on=['Country Name', 'Year'],   #df_country_facts_cleaned
)

df.head()

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag,"Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag",Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag,Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag,"Population, total [SP.POP.TOTL]_missing_flag","Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag",Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag,Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag"
0,South_Sudan_South_Sudan,South Sudan,South Sudan,6.7984,33.1308,103.426655,276.213349,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,South_Sudan_Lakes,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,South_Sudan_Rumbek_East,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Filter rows where ANY column is NaN

cols_to_Use = ['Country Name', 'Year','Population, total [SP.POP.TOTL]',
       'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]',
       'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]',
       'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]',
       'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]',
       'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]',
       'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]',
       'Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]',
       'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]',
       'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]',
       'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]',
       'Permanent cropland (% of land area) [AG.LND.CROP.ZS]',
       'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]',
       'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]',
       'Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]_missing_flag',
       'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]_missing_flag',
       'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]_missing_flag',
       'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]_missing_flag',
       'Country Code_missing_flag',
       'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag',
       'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag',
       'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag',
       'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag',
       'Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag',
       'Population, total [SP.POP.TOTL]_missing_flag',
       'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag',
       'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag',
       'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag',
       'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag']
print(df.columns)



Index(['id', 'name', 'country', 'latitude', 'longitude', 'minBorderDistanceKm',
       'minCapitalDistanceKm', 'periodStart', 'periodEnd', 'LocalEventCount',
       'LocalDistinctActorCount', 'LocalDistinctEventTypes',
       'LocalDistinctEventSubtypes', 'LocalTotalFatalities',
       'LocalAvgSeverity', 'LocalProtestersEventCount',
       'LocalStateForcesEventCount', 'LocalPoliticalMilitiaEventCount',
       'LocalIdentityMilitiaEventCount', 'LocalRebelGroupEventCount',
       'LocalRiotersEventCount', 'LocalCiviliansEventCount',
       'LocalOtherTypeEventCount', 'RegionalEventCount',
       'RegionalDistinctActorCount', 'RegionalDistinctEventTypes',
       'RegionalDistinctEventSubTypes', 'RegionalTotalFatalities',
       'RegionalAvgSeverity', 'RegionalProtestersEventCount',
       'RegionalStateForcesEventCount', 'RegionalPoliticalMilitiaEventCount',
       'RegionalIdentityMilitiaEventCount', 'RegionalCiviliansEventCount',
       'RegionalRebelGroupEventCount', 'RegionalRioters

In [21]:
rows_with_nan = df[df[cols_to_use].isna().any(axis=1)]
print('Rows with at least one NaN in the country fact columns: ', len(rows_with_nan))
rows_with_nan

#Delete rows with a NaN in the country fact data: in practice this is 2015 only. 
#Commented out as we may want those to add ARIMA style lag features. 
#indexes_with_nan =  df[df[cols_to_use].isna().any(axis=1)].index
#df = df[~rows_with_nan]

Rows with at least one NaN in the country fact columns:  233296


,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag,"Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag",Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag,Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag,"Population, total [SP.POP.TOTL]_missing_flag","Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag",Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag,Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag"
0,South_Sudan_South_Sudan,South Sudan,South Sudan,6.7984,33.1308,103.426655,276.213349,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,South_Sudan_Lakes,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,South_Sudan_Rumbek_East,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233291,Sudan_Rama,Rama,Sudan,NaN,NaN,NaN,NaN,2015-12-31,2016-01-28,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233292,Sudan_Ndjamena,Ndjamena,Sudan,NaN,NaN,NaN,NaN,2015-12-31,2016-01-28,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233293,Kenya_Koma,Koma,Kenya,NaN,NaN,NaN,NaN,2015-12-31,2016-01-28,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233294,Kenya_Kapchorwa,Kapchorwa,Kenya,NaN,NaN,NaN,NaN,2015-12-31,2016-01-28,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:


# Extract unique (country, Year) pairs
unique_nan_combos = (
    rows_with_nan[['Country Name', 'Year']]
    .drop_duplicates()
)

print(sorted(unique_nan_combos))

#drop the rogue Year column and Country Name- just required for joining. 
df = df.drop(columns=['Year', 'Country Name'])


['Country Name', 'Year']


In [23]:
print("Values with missing country fact data:")
print(unique_nan_combos)

Values with missing country fact data:
  Country Name  Year
0          NaN  2015


In [24]:
del df_country_facts_cleaned

## Merging in Election Data

In [25]:
df_elections_cleaned = pd.read_csv("Election Data original\elections.csv")

In [26]:
# format the date column
df_elections_cleaned['Date'] = pd.to_datetime(df_elections_cleaned['Date'], format='%d/%m/%Y')

#Unique periods by country
tmp = df[['country', 'periodStart', 'periodEnd']].drop_duplicates()

#merge elections into tmp
tmp = tmp.merge(
    df_elections_cleaned[['Country', 'Date']],
    how='left',
    left_on='country',
    right_on='Country'
)

#create is_voting before grouping
tmp['is_voting'] = (
    (tmp['Date'] >= tmp['periodStart']) &
    (tmp['Date'] <= tmp['periodEnd'])
).astype(int)

#Collapse multiple elections per period
tmp = (
    tmp.groupby(['country', 'periodStart', 'periodEnd'])['is_voting']
       .max()
       .reset_index()
)

#merge back to df
df = df.merge(
    tmp,
    how='left',
    on=['country', 'periodStart', 'periodEnd']
)

del tmp


In [27]:
#Convert to int and show:
pd.to_numeric(df['is_voting'], errors = 'raise') #throw error, will need to fix if triggered. 

#Show is_voting being applied successfully
df[df["is_voting"] == True].head(10)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag,"Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag",Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag,Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag,"Population, total [SP.POP.TOTL]_missing_flag","Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag",Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag,Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag",is_voting
83332,Ethiopia_Ethiopia,Ethiopia,Ethiopia,10.0052,38.9820,245.253400,117.971198,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83333,Ethiopia_Oromia,Oromia,Ethiopia,9.8562,37.0230,195.349154,211.256409,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83334,Ethiopia_West_Wellega,West Wellega,Ethiopia,9.2756,35.6549,167.353113,339.312139,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83335,Ethiopia_Mana_Sibu,Mana Sibu,Ethiopia,9.8265,35.2488,105.255705,393.562875,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83336,Ethiopia_Benguwa,Benguwa,Ethiopia,9.7785,34.9205,74.354328,427.525551,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83361,Ethiopia_East_Wellega,East Wellega,Ethiopia,9.5647,36.6320,233.887306,239.334564,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83362,Ethiopia_Gobu_Seyo,Gobu Seyo,Ethiopia,9.0932,36.9577,280.137873,194.964259,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83363,Ethiopia_Ago,Ago,Ethiopia,9.1500,36.9500,274.055708,196.317254,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83367,Ethiopia_Amhara,Amhara,Ethiopia,10.0052,38.9820,245.253400,117.971198,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
83368,Ethiopia_South_Wello,South Wello,Ethiopia,10.4709,38.8111,196.538626,166.699942,2015-05-21,2015-06-18,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


## Deleting places with 0 local events

Places with 0 local events are likely to have been created as admin3-1 places. Them having 0 local events does not make them peaceful, they are likely places further up the WITHIN chain with conflict events being recorded lower down. 

### Deletion of places with no location data
These places will have been created as admin3-1 places, with events further down the WITHIN chain in the knowledge graph. They are removed to stop them being used as training data- there being 0 conflict events in these places does not mean they are peaceful places. Their events are instead captured in lower level places. 

In [28]:
unique_ids = df["id"].unique()
print("Unique places: ", len(unique_ids))

print("Observations before remove missing those with missing spatial information: ", df.shape[0])
unique_ids = df[df["latitude"].isna() | df["longitude"].isna() ]["id"].unique()
#print("Ids with no spatial information: ", unique_ids)
df = df[~df["id"].isin(unique_ids)] #~ is negation: rows not in the set.
print("Observations after remove missing those with missing spatial information: ", df.shape[0])


unique_ids = df["id"].unique()
print("Unique places: ", len(unique_ids))


Unique places:  16652
Observations before remove missing those with missing spatial information:  2332960
Observations after remove missing those with missing spatial information:  2306640
Unique places:  16464


In [29]:
ids_all_zero = (
    df.groupby("id")["LocalEventCount"]
      .apply(lambda s: (s == 0).all())
      .loc[lambda x: x]        # keep only True groups
      .index
      .tolist()
)
print("Count of IDs with 0 Local Events: ", len(ids_all_zero))
#print(ids_all_zero)


Count of IDs with 0 Local Events:  2904


In [30]:
unique_ids = df["id"].unique()
print("Unique places: ", len(unique_ids))

print("Observations before remove missing those with 0 local events: ", df.shape[0])
ids_all_zero
#print("Ids with no spatial information: ", unique_ids)
df = df[~df["id"].isin(ids_all_zero)] #~ is negation: rows not in the set.
print("Observations before remove missing those with 0 local events: ", df.shape[0])

unique_ids = df["id"].unique()
print("Unique places: ", len(unique_ids))

Unique places:  16464
Observations before remove missing those with 0 local events:  2306640
Observations before remove missing those with 0 local events:  1899800
Unique places:  13560


## Deletion of the Final Period
The final period is deleted, the odds are strong that this is not a complete period given data can only be loaded up to the contents of the ACLED data.


In [31]:
max_period_start = max(df['periodStart'])
print("Max periodStart:", max_period_start)
print("Observations before removing last period: ", df.shape[0])
df = df[df["periodStart"] != max_period_start]
print("Observations after removing last period ", df.shape[0])

Max periodStart: 2025-08-28 00:00:00
Observations before removing last period:  1899800
Observations after removing last period  1886230


## Split into Training and Testing Sets and Write to .CSV
the training set is the final 12 periods. We will not remove any columns from that, since we need the original values to determine correctness of any model. 

In [54]:
testing_period_count = 12
# Use unique periods, then sort
unique_periods = sorted(df['periodStart'].unique())
training_periods = unique_periods[:-testing_period_count] # all but the last 12
testing_periods  = unique_periods[-testing_period_count:] # last 12


In [57]:
len(training_periods), len(testing_periods)

(127, 12)

In [61]:
df_train = df[df["periodStart"].isin(training_periods)]
df_train.to_csv("Dataset Preprocessed/eastafrica3_2026-09-20_Train.csv", index=False, mode='w+')

counts = df_train.groupby("id")["periodStart"].nunique()
print("train counts: ", counts)

del df_train

train counts:  id
Central_African_Republic_Aba              127
Central_African_Republic_Abagba_2         127
Central_African_Republic_Abba             127
Central_African_Republic_Abba-Bogani      127
Central_African_Republic_Agoudou-Manga    127
                                         ... 
Uganda_Zeu                                127
Uganda_Zirobwe                            127
Uganda_Zoka                               127
Uganda_Zoka_Forest                        127
Uganda_Zombo                              127
Name: periodStart, Length: 13560, dtype: int64


In [62]:
df_test = df[df["periodStart"].isin(testing_periods)]
df_test.to_csv("Dataset Preprocessed/eastafrica3_2026-09-20_Test.csv", index=False, mode='w+')

counts = df_test.groupby("id")["periodStart"].nunique()
print("train counts: ", counts)

del df_test


train counts:  id
Central_African_Republic_Aba              12
Central_African_Republic_Abagba_2         12
Central_African_Republic_Abba             12
Central_African_Republic_Abba-Bogani      12
Central_African_Republic_Agoudou-Manga    12
                                          ..
Uganda_Zeu                                12
Uganda_Zirobwe                            12
Uganda_Zoka                               12
Uganda_Zoka_Forest                        12
Uganda_Zombo                              12
Name: periodStart, Length: 13560, dtype: int64


### End of Notebook. 